In [1]:
import pandas as pd
import numpy as np 
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df= pd.read_csv("Datos-1.csv")
df.describe()

,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
count,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000
mean,79.493671,5.375734,0.047885,0.846137,0.991046,0.630259,0.428615,0.143422,0.237296,2.098977
std,45.754363,1.145010,0.017146,0.403121,0.272369,0.247078,0.150693,0.120034,0.126685,0.553550
min,1.000000,2.839000,0.018480,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.328580
25%,40.250000,4.526000,0.037268,0.545808,0.856823,0.439185,0.328330,0.061675,0.150553,1.759410
50%,79.500000,5.232500,0.043940,0.910245,1.029510,0.696705,0.435515,0.107220,0.216130,2.095415
75%,118.750000,6.243750,0.052300,1.158448,1.214405,0.811013,0.549092,0.180255,0.309883,2.462415
max,158.000000,7.587000,0.136930,1.690420,1.402230,1.025250,0.669730,0.551910,0.795880,3.602140


Pregunta 1

“¿Cuánto ‘compra’ la felicidad el dinero?”
Haz un scatter plot entre Economy (GDP per Capita) y Happiness Score con línea de tendencia. Cuenta si el retorno marginal parece decreciente y destaca un outlier.

In [3]:
r1=df.groupby("Economy (GDP per Capita)")["Happiness Score"].mean().reset_index().round(2)
fig=px.scatter(df, x="Economy (GDP per Capita)", y="Happiness Score", trendline= "ols")
fig.show()

pregunta 2

“Top 10: ¿liderazgo estable o frágil?”
Muestra el Top-10 países con mejor Happiness Rank en un gráfico de líneas ordenado. Narra si hay “brechas de oro” entre los primeros tres y el resto.

In [4]:
r2= df.head(10).sort_values("Happiness Score",ascending=False)
fig2= px.line(r2, x="Country", y="Happiness Score",markers=True ,title="Top 10 paises con mejor Happiness score")
fig2.show()

pregunta3

“¿Qué región ‘sonríe’ más?”
Construye un bar chart con el promedio de Happiness Score por Region. Cuenta qué región lidera y cuál queda rezagada. Incluye barras de error (Standard Error).

In [5]:
r3= df.groupby("Region")["Happiness Score"].mean().reset_index().round(2)
r4= df.groupby("Region")["Standard Error"].mean().reset_index().round(2)
r5= pd.merge(r3,r4, on="Region",how="inner")
r5

fig3= px.bar(r5,x="Region",y="Happiness Score", error_y="Standard Error")
fig3.show()

pregunta 4

“Anatomía de un país feliz”
Para el país #1 en Happiness Rank, arma un stacked bar con los factores (Economy, Family, Health, Freedom, Trust, Generosity). Explica qué pilar pesa más.

In [6]:
r6= df.head(1).sort_values("Happiness Rank",ascending=False)
factors = ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
           "Freedom", "Trust (Government Corruption)", "Generosity"]
r7 = r6.melt(id_vars=["Country"], value_vars=factors, var_name="Factor", value_name="Valor")

# Gráfico stacked bar horizontal
fig4 = px.bar(r7, x="Valor", y="Country", color="Factor", orientation="h", title=f"Descomposición de factores para {r6.iloc[0]['Country']} (Rank 1)", text="Valor")

fig4.update_traces(texttemplate="%{text:.2f}", textposition="inside")
fig4.show()

### Pregunta 5

In [7]:
r8 = df.copy()
r8[factors] = (df[factors] - df[factors].min()) / (df[factors].max() - df[factors].min())

# Heatmap países vs factores
fig5 = px.imshow(r8[factors],
                x=factors,
                y=df["Country"],
                color_continuous_scale="Viridis",
                title="Mapa de calor: Factores normalizados por país")

fig5.update_layout(yaxis=dict(title="País", automargin=True),
                  xaxis_title="Factores")
fig5.show()


### Pregunta 6
“Hermandades ocultas entre países”
Organiza países por similitud y muestra un heatmap clusterizado. Cuenta qué dos países resultan “hermanos” inesperados.

In [22]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage
import plotly.figure_factory as ff

# --- Preprocesamiento ---
r6 = df.copy()
r6.drop(['Happiness Rank', 'Happiness Score', 'Region', 'Standard Error'], axis=1, inplace=True)

# Usamos el país como índice
r6.set_index('Country', inplace=True)
r6 = r6.select_dtypes(include=['float64', 'int64'])

# Normalizamos todas las variables restantes
scaler = StandardScaler()
df_normalizado = pd.DataFrame(
    scaler.fit_transform(r6),
    columns=r6.columns,
    index=r6.index
)

# --- Clustering jerárquico ---
data_matrix = df_normalizado.values
linked_matrix = linkage(data_matrix, method='average', metric='euclidean')

# --- Dendrograma con heatmap clusterizado ---
fig = ff.create_dendrogram(
    df_normalizado,
    orientation='right',
    labels=df_normalizado.index.tolist(),
)

# Reordenamos el heatmap según el dendrograma
for i in range(len(fig['data'])):
    if fig['data'][i]['type'] == 'heatmap':
        reordered_countries = fig['layout']['yaxis']['ticktext']

        fig['data'][i].update(
            z=df_normalizado.loc[reordered_countries, :].values,
            x=df_normalizado.columns.tolist(),
            y=reordered_countries,
            colorscale='RdBu',
            colorbar=dict(title='Valor Normalizado (Z-Score)'),
            hovertemplate='Factor: %{x}<br>País: %{y}<br>Z-Score: %{z}<extra></extra>'
        )

# --- Ajustes de layout ---
fig.update_layout(
    title={
        'text': 'Hermandades Ocultas: Mapa de Calor Clusterizado de Perfiles de Felicidad',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20}
    },
    width=1200,
    height=1000,
    margin={'r': 200},
    xaxis_title='Factores de Felicidad',
    yaxis=dict(tickangle=0, tickfont={'size': 8})
)

fig.show()


El análisis de similitud revela “hermandades” inesperadas: por ejemplo, un país latinoamericano como Costa Rica aparece cercano a Nueva Zelanda, pese a pertenecer a contextos geográficos y económicos distintos.
Esto sugiere que factores sociales y de calidad de vida pueden acercar realidades nacionales muy diferentes, generando paralelismos ocultos más allá de la economía pura.

### Pregunta 7

In [9]:
fig = px.histogram(df,
                   x="Happiness Score",
                   marginal="box",
                   nbins=20,
                   title="Distribución global de Happiness Score con boxplot marginal",
                   hover_data=["Country", "Region"])

fig.update_layout(xaxis_title="Happiness Score",
                  yaxis_title="Número de países")
fig.show()

In [10]:
# Separar LATAM y resto
latam = df[df["Region"] == "Latin America and Caribbean"]
resto = df[df["Region"] != "Latin America and Caribbean"]

# Histograma overlay
fig = px.histogram(resto, x="Happiness Score", nbins=20, opacity=0.5,
                   title="Distribución de Happiness Score: Mundo vs LATAM")
fig.add_histogram(x=latam["Happiness Score"], nbinsx=20, opacity=0.7, name="LATAM")



fig.update_layout(barmode="overlay",
                  xaxis_title="Happiness Score",
                  yaxis_title="Número de países")
fig.show()

In [11]:
fig = px.histogram(df,
                   x="Happiness Score",
                   color="Region",
                   marginal="box",
                   nbins=20,
                   title="Distribución global de Happiness Score con LATAM resaltado",
                   hover_data=["Country"])

fig.update_layout(xaxis_title="Happiness Score",
                  yaxis_title="Número de países")
fig.show()

### Pregunta 8

In [12]:
fig = px.box(df,
             x="Region",
             y="Happiness Score",
             points="all",  # muestra también los puntos individuales
             title="Distribución de Happiness Score por Región",
             hover_data=["Country"])

fig.update_layout(xaxis_title="Región",
                  yaxis_title="Happiness Score",
                  xaxis_tickangle=-45)  # girar etiquetas si son largas
fig.show()

### Pregunta 9

In [13]:
fig=px.scatter(df, x="Trust (Government Corruption)", y="Freedom", trendline= "ols", marginal_x="histogram", marginal_y="histogram", color="Region")
fig.show()

### Pregunta 10
“Violines de felicidad”
px.violin de Happiness Score por Region (con box=True). Cuenta qué regiones muestran mayor dispersión y si la mediana cambia tu lectura del “ranking” simple. Pide resaltar LATAM con category_orders y anotación breve.

In [14]:
category_order = {"Region": ["Latin America and Caribbean"] + 
                  sorted(r for r in df["Region"].unique() if r != "Latin America and Caribbean")}

fig = px.violin(
    df,
    x="Region",
    y="Happiness Score",
    box=True,
    points="outliers",
    category_orders=category_order,
    hover_data=["Country", "Happiness Rank"]
)

# Personalizar título y ejes
fig.update_layout(
    title="Distribución de Happiness Score por Región",
    xaxis_title="Región",
    yaxis_title="Happiness Score",
    xaxis_tickangle=-45
)

# Anotación en LATAM (mediana e IQR)
latam = df[df["Region"] == "Latin America and Caribbean"]["Happiness Score"]
fig.add_annotation(
    x="Latin America and Caribbean",
    y=latam.median(),
    text=f"Mediana LATAM: {latam.median():.2f}<br>IQR: {(latam.quantile(0.75)-latam.quantile(0.25)):.2f}",
    showarrow=True,
    arrowhead=2,
    ay=-40
)

fig.show()


Las regiones muestran distinta dispersión: Medio Oriente y Norte de África y Europa Occidental son las más variables.
Al usar la mediana en lugar de la media, el ranking cambia en algunas regiones, lo que revela la influencia de valores extremos. 
LATAM aparece con una dispersión moderada, y su mediana refleja una posición intermedia en la distribución global.

### PRegunta 11
“¿Generosidad: vitamina o decorado?”
px.bar con el promedio de Generosity por Region, ordenado descendentemente. Destaca 2 países líderes en generosidad con una segunda figura px.bar (Top-10 países). Narra si la generosidad “explica” felicidad o sólo acompaña (apóyate en hover_data con Happiness Score).

In [15]:
region_avg = df.groupby("Region", as_index=False)["Generosity"].mean().round(3)
region_avg = region_avg.sort_values("Generosity", ascending=False)

fig1 = px.bar(
    region_avg,
    x="Region",
    y="Generosity",
    title="Generosidad Promedio por Región",
    labels={"Generosity": "Generosidad Promedio", "Region": "Región"},
    hover_data={"Generosity": True}
)
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

# --- Gráfico 2: Top-10 países más generosos ---
top10 = df.nlargest(10, "Generosity")

fig2 = px.bar(
    top10,
    x="Country",
    y="Generosity",
    title="Top-10 Países en Generosidad",
    labels={"Generosity": "Generosidad", "Country": "País"},
    hover_data=["Happiness Score"]
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()


La generosidad muestra variaciones claras entre regiones y países, con algunos líderes destacados.
Sin embargo, al observar el Happiness Score en hover_data, se nota que altos niveles de generosidad no siempre coinciden con altos niveles de felicidad. 
Más que un motor principal, la generosidad parece funcionar como un 'acompañamiento' de la felicidad en lugar de su explicación central.

### pregunta 12
“Radiografía de correlaciones”
Calcula la matriz de correlación entre Economy, Family, Health, Freedom, Trust, Generosity, Happiness Score y muestra un px.imshow. Señala la relación más fuerte y la más débil; propone una hipótesis de causalidad (con cautela) en 2 frases.



In [16]:
cols = ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
        "Freedom", "Trust (Government Corruption)", "Generosity", "Happiness Score"]

corr = df[cols].corr().round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Radiografía de Correlaciones entre Variables Clave",
    labels={"color": "Correlación"}
)
fig.show()

La relación más fuerte se observa entre Economía y Felicidad, lo que sugiere que el bienestar material pesa mucho en la percepción de felicidad.
En cambio, Generosidad apenas se relaciona, lo que indica que su efecto podría ser más simbólico que causal.

### Pregunta 13
“Rutas categorizadas (bajo/medio/alto)”
Discretiza 3–4 variables (p. ej., Economy, Health, Freedom, Trust) con pd.qcut en Bajo/Medio/Alto y usa px.parallel_categories. Cuenta la ruta modal de países con Happiness Score alto y contrástala con la ruta de puntajes bajos.

In [17]:
vars_disc = ["Economy (GDP per Capita)", "Health (Life Expectancy)", 
             "Freedom", "Trust (Government Corruption)"]

for v in vars_disc:
    df[v + "_cat"] = pd.qcut(df[v], q=3, labels=["Bajo", "Medio", "Alto"])

# --- Clasificar Happiness Score (alto/bajo) ---
df["Felicidad_cat"] = pd.qcut(df["Happiness Score"], q=3, labels=["Bajo", "Medio", "Alto"])

# --- Gráfico de rutas ---
fig = px.parallel_categories(
    df,
    dimensions=[v + "_cat" for v in vars_disc] + ["Felicidad_cat"],
    color=df["Happiness Score"],
    color_continuous_scale="Viridis",
    labels={
        "Economy (GDP per Capita)_cat": "Economía",
        "Health (Life Expectancy)_cat": "Salud",
        "Freedom_cat": "Libertad",
        "Trust (Government Corruption)_cat": "Confianza",
        "Felicidad_cat": "Felicidad"
    },
    title="Rutas categorizadas: Bajo, Medio y Alto"
)
fig.show()

Los países con puntajes altos de felicidad siguen mayoritariamente la ruta Economía Alta → Salud Alta → Libertad Alta → Confianza Media/Alta.
En contraste, los de bajos puntajes se concentran en la ruta Economía Baja → Salud Baja → Libertad Baja → Confianza Baja.
Esto sugiere que la combinación de bienestar material y servicios de salud, acompañados de libertad y cierta confianza institucional, marca la diferencia en los niveles de felicidad.

### Pregunta 14
“Triada crítica: Salud + Familia + Felicidad”
px.scatter_3d con x=Health, y=Family, z=Happiness Score, color por Region. Narra si existe una “meseta” de alta felicidad cuando Salud y Familia superan ciertos umbrales. Pide hover_data con Country y Economy.


In [18]:
fig = px.scatter_3d(
    df,
    x="Health (Life Expectancy)",
    y="Family",
    z="Happiness Score",
    color="Region",
    hover_data=["Country", "Economy (GDP per Capita)"],
    title="Triada crítica: Salud + Familia + Felicidad"
)
fig.show()

### Pregunta 15
“LATAM vs. Mundo: ¿pétalos desbalanceados?”
Compara promedios de factores (Economy, Family, Health, Freedom, Trust, Generosity) LATAM vs Global en px.line_polar o px.bar_polar con dos trazas. Cuenta qué “pétalos” están más abiertos/cerrados en LATAM y cómo eso se refleja en su Happiness Score.

In [19]:
factors = ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", 
           "Freedom", "Trust (Government Corruption)", "Generosity"]

# --- Promedios LATAM y Global ---
latam_avg = df[df["Region"].str.contains("Latin America", case=False)][factors].mean()
world_avg = df[factors].mean()

data = pd.DataFrame({
    "Factor": factors * 2,
    "Valor": list(latam_avg) + list(world_avg),
    "Grupo": ["LATAM"] * len(factors) + ["Global"] * len(factors)
})

# --- Gráfico polar ---
fig = px.line_polar(
    data,
    r="Valor",
    theta="Factor",
    color="Grupo",
    line_close=True,
    title="LATAM vs. Mundo"
)
fig.show()

Los “pétalos” de Economía, Salud y Confianza aparecen más cerrados en LATAM respecto al promedio global, lo que refleja desventajas estructurales que impactan en su puntaje de felicidad.
En cambio, Familia y Generosidad se muestran relativamente más abiertas, sugiriendo que las redes sociales y los valores comunitarios son un contrapeso parcial a las carencias institucionales y materiales.